### This script will take the folder structure of the results of KSA_project_pipeline 2 and match it with those of pipeline 1

In [ ]:
import os
import pandas as pd

# ============================================================
# CONFIG — update these paths/lists as your project grows
# ============================================================

# Root folder that contains all experiment folders
ROOT_DIR = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification"

# All aggregation method folder names (order here = order of row groups in output)
AGGREGATION_METHODS = [
    "Caption_based_aggregation",
    "Averaging",
    "Tissue_Type_Clustering",
    "TITAN"
]

# All model folder names
MODELS = [
    "H-Optimus-1",
    "Conch1_5",
    "UNI2"
]

# Fixed parts of the path after the model folder
FIXED_SUBPATH = os.path.join("1-MSIH", "Output", "cross_valid_avg_eval_metrics.xlsx")

# Output file path
OUTPUT_FILE = os.path.join(ROOT_DIR, "best_of_all_exps_metric.xlsx")

# ============================================================
# HELPER: extract experiment name from folder name
# ============================================================
def get_experiment_name(folder_name: str) -> str:
    if "_Result" in folder_name:
        return folder_name.split("_Result")[0]
    return folder_name 

# ============================================================
# MAIN
# ============================================================
def build_best_of_all_exps_metric(root_dir, aggregation_methods, models, fixed_subpath, output_file):

    # Find all experiment folders (folders ending in _Results)
    experiment_folders = [
        f for f in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, f)) and "_Results" in f
    ]

    if not experiment_folders:
        print(f"[WARNING] No experiment folders found in {root_dir}")
        return

    exp_sheets = {}

    for exp_folder in sorted(experiment_folders):
        exp_name = get_experiment_name(exp_folder)
        print(f"\n{'='*60}")
        print(f"Experiment: {exp_name}")
        print(f"{'='*60}")

        agg_blocks = []

        for agg_method in aggregation_methods:
            print(f"  Aggregation: {agg_method}")
            merged_df = pd.DataFrame()

            for model in models:
                file_path = os.path.join(root_dir, exp_folder, agg_method, model, fixed_subpath)

                if not os.path.exists(file_path):
                    continue

                try:
                    df = pd.read_excel(file_path)

                    if "Metric" not in df.columns:
                        continue

                    # Identify value column
                    if "AvgFolds" in df.columns:
                        value_column = "AvgFolds"
                    elif "Value" in df.columns:
                        value_column = "Value"
                    else:
                        continue

                    # Create a temporary DF with a sort_key based on the original file order
                    temp = df[["Metric", value_column]].copy()
                    temp["Metric"] = temp["Metric"].astype(str).str.strip()
                    
                    # FIX: Create the sort_key from the current range index 
                    # This ensures lin_acc stays at the top and rf_conf_matrix at the bottom
                    temp["sort_key"] = range(len(temp))
                    
                    # Update metric name to include the aggregation suffix
                    temp["Metric"] = temp["Metric"] + f"_{agg_method}"
                    
                    col_name = f"{model}_N"
                    temp.rename(columns={value_column: col_name}, inplace=True)
                    temp.set_index("Metric", inplace=True)

                    if merged_df.empty:
                        merged_df = temp
                    else:
                        # We only want the metric values for the new model column
                        merged_df = merged_df.join(temp[[col_name]], how="outer")

                except Exception as e:
                    print(f"    [ERROR] {file_path}: {e}")

            if not merged_df.empty:
                # Ensure the final block for this aggregation is sorted by the original file order
                merged_df = merged_df.sort_values("sort_key").drop(columns=["sort_key"]).reset_index()
                agg_blocks.append(merged_df)
            else:
                print(f"    [SKIP] No data for '{agg_method}'")

        if agg_blocks:
            combined = pd.concat(agg_blocks, axis=0, ignore_index=True)
            exp_sheets[exp_name] = combined
            print(f"  -> Sheet '{exp_name}' created with {len(combined)} rows.")

    if exp_sheets:
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
            for sheet_name, df in exp_sheets.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"\n✅ Success! File saved to: {output_file}")
    else:
        print("\n[ERROR] No data collected.")

if __name__ == "__main__":
    build_best_of_all_exps_metric(ROOT_DIR, AGGREGATION_METHODS, MODELS, FIXED_SUBPATH, OUTPUT_FILE)


Experiment: TCGA_PAIP_EV
  Aggregation: Caption_based_aggregation
    [SKIP] No data for 'Caption_based_aggregation'
  Aggregation: Averaging
    [SKIP] No data for 'Averaging'
  Aggregation: Tissue_Type_Clustering
    [SKIP] No data for 'Tissue_Type_Clustering'
  Aggregation: TITAN
    [SKIP] No data for 'TITAN'

Experiment: TCGA
  Aggregation: Caption_based_aggregation
  Aggregation: Averaging
  Aggregation: Tissue_Type_Clustering
  Aggregation: TITAN
  -> Sheet 'TCGA' created with 120 rows.

✅ Success! File saved to: D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\best_of_all_exps_metric.xlsx


### Below script will generate radar plots from the best of all exps metrics excel file

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import os

# ============================================================
# CONFIG — update these as your project grows
# ============================================================

EVAL_METRIC = "BalAcc"  # BalAcc or AUC

INPUT_FILE  = r"C:\Projects\Other Projects\KSA_Project\Slide_Classification\best_of_all_exps_metric.xlsx"
OUTPUT_DIR  = r"C:\Projects\Other Projects\KSA_Project\Slide_Classification\Radar_Plots"

# Aggregation methods — order determines the slice layout (starts from top, clockwise)
AGGREGATION_METHODS = [
    "Caption_based_aggregation",
    "Averaging",
    "Tissue_Type_Clustering"
]

# All model folder names
FOUNDATION_MODELS = ["H-Optimus-1", "Conch1_5", "UNI2"]
MODEL_COLORS      = ["#e6194b", "#3cb44b", "#4363d8"]

# Background colors (dynamically mapped)
AGGREGATION_COLORS = {
    "Caption_based_aggregation": "#d0e0ff",
    "Averaging":                  "#ffe0e0",
    "Tissue_Type_Clustering":     "#c0ffc0"
}

# Classifier models names on the Radar plot
CLASSIFIER_MODELS = ["Lin", "ANN", "KNN", "Proto", "RF"]
CLASSIFIER_LABEL_MAP = {
    "Lin": "Linear", 
    "ANN": "ANN", 
    "KNN": "KNN", 
    "Proto": "Proto", 
    "RF": "RF"
}

MODEL_COLUMN_MAP = {
    "H-Optimus-1": "H-Optimus-1_N",
    "Conch1_5":    "Conch1_5_N",
    "UNI2":        "UNI2_N"
}

CLASSIFIER_EXCEL_PREFIX = {
    "Lin": "lin", 
    "ANN": "ann", 
    "KNN": "knn", 
    "Proto": "proto", 
    "RF": "r_forest"
}

EVAL_METRIC_EXCEL_SUFFIX = {
    "BalAcc": "bacc", 
    "AUC": "auroc", 
    "Acc": "acc", 
    "MacroF1": "macro_f1", 
    "WF1": "weighted_f1",
}

# ============================================================
# HELPERS
# ============================================================
def close_circle(data_list):
    return data_list + [data_list[0]]

def extract_values(df, experiment_name, aggregation_methods, classifier_models,
                   eval_metric, foundation_models, model_column_map):
    df["Metric"] = df["Metric"].astype(str).str.strip()
    axis_labels, agg_groups = [], []
    models_data = {fm: [] for fm in foundation_models}

    for agg in aggregation_methods:
        for clf in classifier_models:
            full_clf = CLASSIFIER_LABEL_MAP.get(clf, clf)
            axis_labels.append(f"{full_clf}-{eval_metric}\n({agg})")
            agg_groups.append(agg)
            clf_prefix = CLASSIFIER_EXCEL_PREFIX.get(clf, clf.lower())
            metric_suffix = EVAL_METRIC_EXCEL_SUFFIX.get(eval_metric, eval_metric.lower())
            metric_row = f"{clf_prefix}_{metric_suffix}_{agg}"
            row = df[df["Metric"] == metric_row]

            for fm in foundation_models:
                col = model_column_map[fm]
                if row.empty or col not in df.columns:
                    models_data[fm].append(0.0)
                else:
                    val = row[col].iloc[0]
                    models_data[fm].append(round(float(val), 4) if pd.notna(val) else 0.0)
    return axis_labels, agg_groups, models_data

# ============================================================
# MAIN
# ============================================================
def generate_radar_plots(input_file, output_dir, aggregation_methods, aggregation_colors,
                         classifier_models, foundation_models, model_colors,
                         model_column_map, eval_metric):

    os.makedirs(output_dir, exist_ok=True)
    xl = pd.ExcelFile(input_file)

    for exp_name in xl.sheet_names:
        df = pd.read_excel(input_file, sheet_name=exp_name)
        axis_labels, agg_groups, models_data = extract_values(
            df, exp_name, aggregation_methods, classifier_models,
            eval_metric, foundation_models, model_column_map
        )

        N = len(axis_labels)
        angle_step = 2 * np.pi / N
        half_step  = angle_step / 2
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False) + half_step
        angles_closed = np.concatenate((angles, [angles[0]]))

        fig = plt.figure(figsize=(24, 24))
        ax  = fig.add_subplot(111, polar=True)
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        # -------------------------------------------------------
        # DYNAMIC Background fills — Adjusts to any number of Agg Methods
        # -------------------------------------------------------
        n_groups = len(aggregation_methods)
        group_span = 2 * np.pi / n_groups 
        max_r = 1.05

        for i, agg in enumerate(aggregation_methods):
            theta_start = i * group_span
            theta_end   = (i + 1) * group_span
            theta_arc   = np.linspace(theta_start, theta_end, 200)
            theta_wedge = np.concatenate(([theta_start], theta_arc, [theta_end]))
            r_wedge     = np.concatenate(([0], np.full(len(theta_arc), max_r), [0]))
            ax.fill(theta_wedge, r_wedge, color=aggregation_colors.get(agg, "#f0f0f0"), alpha=0.4, zorder=0)

        # -------------------------------------------------------
        # Plot each foundation model
        # -------------------------------------------------------
        for (fm, values), color in zip(models_data.items(), model_colors):
            stats = close_circle(values)
            ax.plot(angles_closed, stats, color=color, label=fm, linewidth=3, marker='o', markersize=6, zorder=5)
            ax.fill(angles_closed, stats, color=color, alpha=0.06, zorder=4)

        # -------------------------------------------------------
        # FIX 2: Axis tick labels
        # -------------------------------------------------------
        ax.set_xticks(angles)
        ax.set_xticklabels([]) 

        for angle, label, agg in zip(angles, axis_labels, agg_groups):
            ax.text(angle, 1.18, label, ha="center", va="center", fontsize=14, fontweight="bold",
                    rotation=0, bbox=dict(facecolor=aggregation_colors.get(agg, "#f0f0f0"), 
                    edgecolor="black", boxstyle="square,pad=0.5", linewidth=1.5))

        # -------------------------------------------------------
        # FIX 4: Scale numbers along EVERY spoke
        # -------------------------------------------------------
        ax.set_yticklabels([])
        ax.set_ylim(0.0, 1.0)
        r_ticks = np.arange(0.45, 0.91, 0.1)
        ax.set_yticks(r_ticks)

        for angle in angles:
            for r in r_ticks:
                ax.text(angle, r, f"{r:.2f}", ha="center", va="center", fontsize=16, 
                        color="grey", fontweight="bold", zorder=6,
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1.0))

        # -------------------------------------------------------
        # FIX 1: Legends
        # -------------------------------------------------------
        model_handles, model_labels = ax.get_legend_handles_labels()
        fig.legend(model_handles, model_labels, loc="upper right", bbox_to_anchor=(0.98, 0.98),
                   title="Models", fontsize=18, title_fontsize=20, borderpad=1.2,
                   framealpha=0.95, edgecolor="black", fancybox=True, shadow=True)

        agg_handles = [mpatches.Patch(facecolor=aggregation_colors.get(agg, "#f0f0f0"), 
                       edgecolor="black", label=agg) for agg in aggregation_methods]
        fig.legend(handles=agg_handles, loc="upper left", bbox_to_anchor=(0.02, 0.98),
                   title="Aggregation Methods", fontsize=18, title_fontsize=20, borderpad=1.2,
                   framealpha=0.95, edgecolor="black", fancybox=True, shadow=True)

        plt.title(f"Radar Plot — {exp_name} | Metric: {eval_metric}", y=1.12, fontsize=22, fontweight="bold")
        save_path = os.path.join(output_dir, f"{exp_name}_{eval_metric}_radar.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.5)
        plt.close()
        print(f"  ✅ Saved: {save_path}")

if __name__ == "__main__":
    generate_radar_plots(INPUT_FILE, OUTPUT_DIR, AGGREGATION_METHODS, AGGREGATION_COLORS,
                         CLASSIFIER_MODELS, FOUNDATION_MODELS, MODEL_COLORS,
                         MODEL_COLUMN_MAP, EVAL_METRIC)

  ✅ Saved: C:\Projects\Other Projects\KSA_Project\Slide_Classification\Radar_Plots\TCGA_BalAcc_radar.png


## Old version below

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import os

# ============================================================
# CONFIG — update these as your project grows
# ============================================================

EVAL_METRIC = "BalAcc"  # BalAcc or AUC

INPUT_FILE  = r"C:\Projects\Other Projects\KSA_Project\Slide_Classification\best_of_all_exps_metric.xlsx"
OUTPUT_DIR  = r"C:\Projects\Other Projects\KSA_Project\Slide_Classification\Radar_Plots"

# Aggregation methods — order determines the quarter layout (top, right, bottom, left)
AGGREGATION_METHODS = [
    "Caption-Based-Aggregation",
    "Averaging",
    "Tissue-type-clustering",
    "TITAN"
]

# Background colors for each quarter (one per aggregation method)
AGGREGATION_COLORS = {
    "Caption-Based-Aggregation": "#e0e0ff",  # light blue
    "Averaging":                  "#ffe0e0",  # light red
    "Tissue-type-clustering":     "#d0ffd0",  # light green
    "TITAN":                      "#fff5e0",  # light orange
}

# Classifier models to show on the radar axes
CLASSIFIER_MODELS = ["Lin", "ANN", "KNN", "Proto"]
CLASSIFIER_LABEL_MAP = {
    "Lin":   "Linear",
    "ANN":   "ANN",
    "KNN":   "KNN",
    "Proto": "Proto"
}

# Foundation models (lines on the radar) — must match column names in Excel (without _N suffix)
FOUNDATION_MODELS = ["H-Optimus-1", "Conch1_5", "UNI2-h"]
MODEL_COLORS      = ["#e6194b", "#3cb44b", "#4363d8"]  # red, green, blue

# Model column mapping: model name -> Excel column name
MODEL_COLUMN_MAP = {
    "H-Optimus-1": "H-Optimus-1_N",
    "Conch1_5":    "Conch1_5_N",
    "UNI2-h":      "UNI2-h_N"
}

# Maps classifier display name -> Excel metric prefix
CLASSIFIER_EXCEL_PREFIX = {
    "Lin":   "lin",
    "ANN":   "ann",
    "KNN":   "knn",
    "Proto": "proto",
}

# Maps eval metric display name -> Excel metric suffix
EVAL_METRIC_EXCEL_SUFFIX = {
    "BalAcc":  "bacc",
    "AUC":     "auroc",
    "Acc":     "acc",
    "MacroF1": "macro_f1",
    "WF1":     "weighted_f1",
}

# Metric rename map for display labels on radar axes
METRIC_RENAME_MAP = {
    "lin_acc": "Lin_Acc", "lin_bacc": "Lin_BalAcc", "lin_auroc": "Lin_AUC",
    "lin_macro_f1": "Lin_MacroF1", "lin_weighted_f1": "Lin_WF1",
    "ann_acc": "ANN_Acc", "ann_bacc": "ANN_BalAcc", "ann_auroc": "ANN_AUC",
    "ann_macro_f1": "ANN_MacroF1", "ann_weighted_f1": "ANN_WF1",
    "knn_acc": "KNN_Acc", "knn_bacc": "KNN_BalAcc", "knn_auroc": "KNN_AUC",
    "knn_macro_f1": "KNN_MacroF1", "knn_weighted_f1": "KNN_WF1",
    "proto_acc": "Proto_Acc", "proto_bacc": "Proto_BalAcc", "proto_auroc": "Proto_AUC",
    "proto_macro_f1": "Proto_MacroF1", "proto_weighted_f1": "Proto_WF1",
}


# ============================================================
# HELPER: close radar polygon
# ============================================================
def close_circle(data_list):
    return data_list + [data_list[0]]


# ============================================================
# HELPER: extract ordered values from a flat DataFrame
# ============================================================
def extract_values(df, experiment_name, aggregation_methods, classifier_models,
                   eval_metric, foundation_models, model_column_map):
    """
    Returns:
        axis_labels      : list of str  — radar axis tick labels
        agg_groups       : list of str  — which aggregation each axis belongs to
        models_data      : dict {model_name: [float, ...]}
    """
    df["Metric"] = df["Metric"].astype(str).str.strip()

    axis_labels = []
    agg_groups  = []
    models_data = {fm: [] for fm in foundation_models}

    for agg in aggregation_methods:
        for clf in classifier_models:
            full_clf = CLASSIFIER_LABEL_MAP[clf]
            # Axis label shown on radar
            axis_labels.append(f"{full_clf}-{eval_metric}\n({agg})")
            agg_groups.append(agg)

            # Build the correct metric row name matching the Excel data
            clf_prefix = CLASSIFIER_EXCEL_PREFIX.get(clf, clf.lower())
            metric_suffix = EVAL_METRIC_EXCEL_SUFFIX.get(eval_metric, eval_metric.lower())
            metric_row = f"{clf_prefix}_{metric_suffix}_{agg}"

            row = df[df["Metric"] == metric_row]

            for fm in foundation_models:
                col = model_column_map[fm]
                if row.empty or col not in df.columns:
                    models_data[fm].append(0.0)
                else:
                    val = row[col].iloc[0]
                    models_data[fm].append(round(float(val), 4) if pd.notna(val) else 0.0)

    return axis_labels, agg_groups, models_data


# ============================================================
# MAIN: generate one radar plot per experiment sheet
# ============================================================
def generate_radar_plots(input_file, output_dir, aggregation_methods, aggregation_colors,
                         classifier_models, foundation_models, model_colors,
                         model_column_map, eval_metric):

    os.makedirs(output_dir, exist_ok=True)

    # Load all sheet names = all experiments
    xl = pd.ExcelFile(input_file)
    experiment_sheets = xl.sheet_names
    print(f"Found {len(experiment_sheets)} experiment(s): {experiment_sheets}")

    for exp_name in experiment_sheets:
        print(f"\n{'='*60}")
        print(f"Generating radar plot for experiment: {exp_name}")

        df = pd.read_excel(input_file, sheet_name=exp_name)

        axis_labels, agg_groups, models_data = extract_values(
            df, exp_name, aggregation_methods, classifier_models,
            eval_metric, foundation_models, model_column_map
        )

        # Debug print
        print(f"  Axes: {len(axis_labels)}")
        for fm, vals in models_data.items():
            print(f"  {fm}: {vals}")

        N      = len(axis_labels)
        angle_step = 2 * np.pi / N  # angular spacing between axes
        half_step  = angle_step / 2  # offset to center axes within quarters

        # Offset axes by half_step so they sit centered within each 90° quarter
        # Without offset: axes at 0°, 22.5°, 45°, 67.5° (sitting on the boundary)
        # With offset:    axes at 11.25°, 33.75°, 56.25°, 78.75° (centered in quarter)
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False) + half_step
        angles_closed = np.concatenate((angles, [angles[0]]))

        # -------------------------------------------------------
        # Figure
        # -------------------------------------------------------
        fig = plt.figure(figsize=(24, 24))
        ax  = fig.add_subplot(111, polar=True)
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        # -------------------------------------------------------
        # FIX 3: Quarter background fills — exact 90° wedges
        # Quarter boundaries at exactly 0°, 90°, 180°, 270°
        # (i.e. 0, π/2, π, 3π/2 in internal coords, before offset)
        # -------------------------------------------------------
        n_quarters = len(aggregation_methods)
        quarter_span = 2 * np.pi / n_quarters  # exactly π/2 for 4 quarters
        max_r = 1.05  # extend slightly beyond ylim to cover entire grid area

        for i, agg in enumerate(aggregation_methods):
            # Wedge starts at 0° for Q1, 90° for Q2, etc.
            theta_start = i * quarter_span
            theta_end   = theta_start + quarter_span

            # Build wedge: origin -> arc at max_r -> origin
            theta_arc   = np.linspace(theta_start, theta_end, 300)
            theta_wedge = np.concatenate(([theta_start], theta_arc, [theta_end]))
            r_wedge     = np.concatenate(([0], np.full(len(theta_arc), max_r), [0]))

            color = aggregation_colors.get(agg, "#f0f0f0")
            ax.fill(theta_wedge, r_wedge, color=color, alpha=0.4, zorder=0)

        # -------------------------------------------------------
        # Plot each foundation model
        # -------------------------------------------------------
        for (fm, values), color in zip(models_data.items(), model_colors):
            stats = close_circle(values)
            ax.plot(angles_closed, stats, color=color, label=fm,
                    linewidth=3, marker='o', markersize=6, zorder=5)
            ax.fill(angles_closed, stats, color=color, alpha=0.06, zorder=4)

        # -------------------------------------------------------
        # FIX 2: Axis tick labels — HORIZONTAL, no rotation
        # -------------------------------------------------------
        ax.set_xticks(angles)
        ax.set_xticklabels([])  # We draw manually

        for angle, label, agg in zip(angles, axis_labels, agg_groups):
            bg_color = aggregation_colors.get(agg, "#f0f0f0")
            ax.text(
                angle, 1.18, label,
                ha="center", va="center",
                fontsize=14, fontweight="bold",  # CHANGE THIS: Customize axis label text size here (originally 14)
                rotation=0,  # HORIZONTAL — no tilting
                bbox=dict(facecolor=bg_color, edgecolor="black",
                          boxstyle="square,pad=0.5", linewidth=1.5)
            )

        # -------------------------------------------------------
        # FIX 4: Scale numbers along EVERY spoke (like reference)
        # -------------------------------------------------------
        ax.set_yticklabels([])  # hide default radial labels
        ax.set_ylim(0.0, 1.0)
        r_ticks = np.arange(0.45, 0.91, 0.1)  # 0.45, 0.55, 0.65, 0.75, 0.85
        ax.set_yticks(r_ticks)

        for angle in angles:
            for r in r_ticks:
                ax.text(angle, r, f"{r:.2f}",
                        ha="center", va="center",
                        fontsize=16, color="grey",  # CHANGE THIS: Customize radial numbers (0.45, 0.55 etc.) size here (originally 11)
                        fontweight="bold",
                        zorder=6,
                        bbox=dict(facecolor="white", edgecolor="none",
                                  alpha=0.7, pad=1.0))

        # -------------------------------------------------------
        # FIX 1: Legends — properly positioned & visible in PNG
        # -------------------------------------------------------
        # Foundation model legend — use fig.legend with figure coords
        # Get the line handles from the polar ax
        model_handles, model_labels = ax.get_legend_handles_labels()
        model_legend = fig.legend(
            model_handles, model_labels,
            loc="upper right", bbox_to_anchor=(0.98, 0.98),
            title="Models", fontsize=18, title_fontsize=20, borderpad=1.2,
            framealpha=0.95, edgecolor="black",
            fancybox=True, shadow=True
        )

        # Aggregation method legend — use fig.legend with figure coords
        agg_handles = [
            mpatches.Patch(facecolor=aggregation_colors.get(agg, "#f0f0f0"),
                           edgecolor="black", label=agg)
            for agg in aggregation_methods
        ]
        agg_legend = fig.legend(
            handles=agg_handles,
            loc="upper left", bbox_to_anchor=(0.02, 0.98),
            title="Aggregation Methods", fontsize=18, title_fontsize=20, borderpad=1.2,
            framealpha=0.95, edgecolor="black",
            fancybox=True, shadow=True
        )

        # -------------------------------------------------------
        # Title & save
        # -------------------------------------------------------
        plt.title(f"Radar Plot — {exp_name} | Metric: {eval_metric}",
                  y=1.12, fontsize=22, fontweight="bold")

        save_path = os.path.join(output_dir, f"{exp_name}_{eval_metric}_radar.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.5)
        plt.close()
        print(f"  ✅ Saved: {save_path}")


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    generate_radar_plots(
        input_file          = INPUT_FILE,
        output_dir          = OUTPUT_DIR,
        aggregation_methods = AGGREGATION_METHODS,
        aggregation_colors  = AGGREGATION_COLORS,
        classifier_models   = CLASSIFIER_MODELS,
        foundation_models   = FOUNDATION_MODELS,
        model_colors        = MODEL_COLORS,
        model_column_map    = MODEL_COLUMN_MAP,
        eval_metric         = EVAL_METRIC
    )
